In [1]:
import numpy as np
import pandas as pd
from simpeg.electromagnetics.utils.em1d_utils import Stitched1DModel

In [2]:
df = pd.read_csv("../inversion_results/merged_iteration_14_resistivity_wide.csv")

In [3]:
header = np.array(list(df.keys()))
rho_inds = np.array (['rho' in x for x in header])
depth = np.array([float(x.split('_')[1].split('m')[0]) for x in header[rho_inds]]).astype(float)

In [4]:
thicknesses = np.diff(depth)

In [5]:
n_layer = 16
n_sounding = df.shape[0]

In [6]:
line = np.array([int(val.strip('L').strip('T')) for val in df['Line'].values])

In [7]:
topography = df[['x_wgs84', 'y_wgs84', 'dtm']].values
rho = df[header[rho_inds]].values

In [8]:
model = Stitched1DModel(
    topography=topography,
    physical_property=rho[:,:n_layer].flatten(),
    line=line,
    time_stamp=np.arange(n_sounding),
    thicknesses=thicknesses[:n_layer-1],    
)

In [15]:
model.get_3d_mesh(dx=100, dy=100)
model.get_interpolation_matrix()

/var/folders/4m/sc07bkn154s8jfc7xp164by00000gq/T/ipykernel_9450/3097552345.py:1: UserWarning: Size of the mesh (269463520) will greater than 1e6
  model.get_3d_mesh(dx=100, dy=100)


In [16]:
from verde import distance_mask
mask = distance_mask(
    (topography[:,0], topography[:,1]), 
    maxdist=500, 
    coordinates=(model.mesh_3d.cell_centers[:,0], model.mesh_3d.cell_centers[:,1])
)
# model.mesh_3d.write_UBC('second_path_results/mesh.msh')



In [17]:
models_dict ={}
rho_3d = model.interpolate_from_1d_to_3d(rho[:,:n_layer].flatten())
resistivity_3d = rho_3d.flatten(order='F')
resistivity_3d[~mask] = np.nan
models_dict['rho'] = resistivity_3d
# tmp = np.log10(resistivity_3d.copy())
# tmp[np.isnan(resistivity_3d)] = -1
# model.mesh_3d.write_model_UBC(f'second_path_results/rho_{rho[1]}.mod', tmp)
model.mesh_3d.write_vtk(f'rho_3d.vtr', models=models_dict)    